In [32]:
!pip install google-genai
!pip install httpx

In [35]:
import os
from google import genai
from google.genai import types
import httpx
import time

In [36]:
# 1. Initialize the client
# IMPORTANT: Use environment variables instead of hardcoding your key!
os.environ['GEMINI_API_KEY'] = "AQ." 
client = genai.Client()
model_name = "gemini-2.5-flash-lite"

In [37]:
from google import genai
from google.genai import types
import httpx

client = genai.Client()

file_path = "PHOTOSYNTHESIS.pdf"

# 2. Read the local PDF file bytes directly
with open(file_path, "rb") as f:
    doc_data = f.read()

prompt = "Summarize this document"
start= time.time()
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        types.Part.from_bytes(
            data=doc_data,
            mime_type='application/pdf',
        ),
        prompt
    ]
)
end=time.time()
print(response.text)
total_time_difference= end-start
print(total_time_difference)

This document provides a comprehensive overview of **Photosynthesis**, covering its fundamental definition, cellular machinery, pigments, mechanisms (light and dark reactions), specialized pathways (C3, C4, CAM), influencing factors, and related processes like photorespiration and bacterial photosynthesis.

Here's a summary of the key information:

1.  **Definition and Significance:**
    *   Photosynthesis is the process by which green plants, algae, and some bacteria convert solar energy into chemical energy (carbohydrates) using carbon dioxide (CO2) and water (H2O), releasing oxygen (O2) as a byproduct.
    *   It's a crucial anabolic (building up) process, providing food for nearly all organisms, purifying atmospheric air by removing CO2 and evolving O2, and being essential for the ozone layer.
    *   Globally, 70 billion tonnes of CO2 are fixed, and 170 million tonnes of dry matter are produced annually.

2.  **Cellular Location and Structure:**
    *   The primary sites of photo

In [38]:
# 1. Extract token usage from the response
print(f"Cached tokens: {response.usage_metadata.cached_content_token_count}")
print(f"Input tokens:  {response.usage_metadata.prompt_token_count}")
print(f"Output tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens:  {response.usage_metadata.total_token_count}")

Cached tokens: None
Input tokens:  31481
Output tokens: 1923
Total tokens:  40306


In [39]:
# 2. Upload your PDF using the Files API
pdf_path = "PHOTOSYNTHESIS.pdf" 

print(f"Uploading {pdf_path}...")
document = client.files.upload(
    file=pdf_path, 
    config={'mime_type': 'application/pdf'}
)

# Wait for the file to finish processing
while document.state.name == 'PROCESSING':
    print("Processing file...")
    time.sleep(2)
    document = client.files.get(name=document.name)

print(f"File ready: {document.uri}")

# 3. Create the Cache Manually (Explicit Caching)
system_instruction = "You are an expert at analyzing documents."

print("Creating manual context cache...")
cache = client.caches.create(
    model=model_name,
    config=types.CreateCachedContentConfig(
        system_instruction=system_instruction,
        contents=[document],
        ttl="360s",
        display_name="jupyter_pdf_cache"
    )
)

print(f"Cache created successfully! Cache Name: {cache.name}")

Uploading PHOTOSYNTHESIS.pdf...
File ready: https://generativelanguage.googleapis.com/v1beta/files/436eb8w7bs7u
Creating manual context cache...
Cache created successfully! Cache Name: cachedContents/9c53zm3pckdxof07zjw9upkosqzvplk2eg6obw5m


In [41]:
#META DATA OF CACHED DATA 
for cache in client.caches.list():
  print(cache.name)

cachedContents/9c53zm3pckdxof07zjw9upkosqzvplk2eg6obw5m


In [42]:
# 4. Generate content using the cached tokens
print("Querying the model using the cached PDF...")
start= time.time()
response = client.models.generate_content(
    model=model_name,
    contents="Please provide a detailed summary of this document.",
    config=types.GenerateContentConfig(
        cached_content=cache.name
    )
)
end= time.time()
# 5. Output the results
print("\n" + "="*40)
print("RESPONSE:")
print("="*40)
print(response.text)

total_time_difference= end-start
print(total_time_difference)

Querying the model using the cached PDF...

RESPONSE:
The document discusses photosynthesis, a vital biological process by which green plants, algae, and some bacteria convert light energy into chemical energy in the form of carbohydrates. The process utilizes carbon dioxide (CO2) and water (H2O) as raw materials, with sunlight as the energy source and chlorophyll as the primary pigment that captures light. Oxygen (O2) is released as a byproduct.

Here's a breakdown of the key aspects covered in the document:

**1. Overview of Photosynthesis:**
*   **Definition:** The conversion of solar energy into chemical energy.
*   **Inputs:** Carbon dioxide (CO2), Water (H2O), Light energy.
*   **Outputs:** Carbohydrates (sugars), Oxygen (O2).
*   **Significance:** It forms the base of most food chains, purifies the atmosphere by removing CO2 and releasing O2, and plays a role in maintaining the ozone layer.

**2. The Photosynthetic Apparatus (Chloroplasts):**
*   **Location:** Occurs in chloropl

In [43]:
print("\n" + "="*40)
print("USAGE METADATA (Notice the cached tokens!):")
print("="*40)
print(response.usage_metadata)


USAGE METADATA (Notice the cached tokens!):
cache_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=10
), ModalityTokenCount(
  modality=<MediaModality.DOCUMENT: 'DOCUMENT'>,
  token_count=31476
)] cached_content_token_count=31486 candidates_token_count=1354 candidates_tokens_details=None prompt_token_count=31496 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=20
), ModalityTokenCount(
  modality=<MediaModality.DOCUMENT: 'DOCUMENT'>,
  token_count=31476
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=32850 traffic_type=None


In [44]:
print(f"Cached tokens: {response.usage_metadata.cached_content_token_count}")
print(f"Input tokens:  {response.usage_metadata.prompt_token_count}")
print(f"Output tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens:  {response.usage_metadata.total_token_count}")

Cached tokens: 31486
Input tokens:  31496
Output tokens: 1354
Total tokens:  32850


In [52]:
for cache in client.caches.list():
  print(cache)

In [46]:
# 4. Generate content using the cached tokens
print("Querying the model using the cached PDF...")
start= time.time()
response = client.models.generate_content(
    model=model_name,
    contents="Explain SOLAR SPECTRUM – ELECTROMAGNETIC RADIATION – RADIENT ENERGY",
    config=types.GenerateContentConfig(
        cached_content=cache.name
    )
)
end= time.time()
# 5. Output the results
print("\n" + "="*40)
print("RESPONSE:")
print("="*40)
print(response.text)

total_time_difference= end-start
print(total_time_difference)

Querying the model using the cached PDF...

RESPONSE:
The provided documents contain information about the solar spectrum, electromagnetic radiation, and radiant energy, primarily in the context of photosynthesis. Here's an explanation based on the information presented:

**Solar Spectrum - Electromagnetic Radiation - Radiant Energy**

This section discusses the nature of light and its relevance to photosynthesis.

*   **Light as Waves and Particles:** Light travels as waves, but can also be considered as tiny particles or "energy packets" called photons.
*   **Speed of Light:** Light travels at an incredibly high speed of approximately 3 lakh kilometers per second.
*   **Quantum:** The energy contained within a single photon is called a quantum.
*   **Quantum and Wavelength:** The amount of energy in a quantum is inversely proportional to the wavelength of the light. This means shorter wavelengths have higher energy, and longer wavelengths have lower energy.
*   **Electromagnetic Radi

In [47]:
print(f"Cached tokens: {response.usage_metadata.cached_content_token_count}")
print(f"Input tokens:  {response.usage_metadata.prompt_token_count}")
print(f"Output tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens:  {response.usage_metadata.total_token_count}")

Cached tokens: 31486
Input tokens:  31503
Output tokens: 536
Total tokens:  32039


In [48]:
# DELETING CACHE

In [50]:
from google import genai

client = genai.Client()

# Loop through and delete every active cache
for cache in client.caches.list():
    client.caches.delete(name=cache.name)
    print(f"Successfully deleted: {cache.name}")

In [51]:
for cache in client.caches.list():
  print(cache)